# Web Scraping and Exploratory Data Analysis

---

In this notebook, you'll learn how to **collect data from the web** and then **explore it** using Pandas and visualization tools. We'll scrape book data from [books.toscrape.com](http://books.toscrape.com) — a website built specifically for practicing web scraping — and then perform exploratory data analysis on what we collect.

### What You'll Practice

1. **HTTP Requests** — Fetching web pages with the `requests` library
2. **HTML Parsing** — Extracting structured data with `BeautifulSoup`
3. **Data Cleaning** — Transforming raw scraped text into usable data
4. **Exploratory Data Analysis** — Summarizing, grouping, and visualizing the dataset

### Prerequisites

- Completion of Notebook 01 (Python, Pandas, and stats basics)
- Libraries: `requests`, `beautifulsoup4`, `pandas`, `matplotlib`, `seaborn`

---

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time

%matplotlib inline
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['font.size'] = 12

---

## Part 1: Understanding Web Pages

Before we scrape, let's understand what we're working with. Every web page is made of **HTML** — a structured markup language. When you "view source" on a web page, you see the raw HTML. Our job is to extract the useful information from that structure.

### 1.1 Fetching a Page

The `requests` library lets us download web pages. The response contains the HTML content.

In [ ]:
# Fetch the main page of our target site
url = "http://books.toscrape.com/"
response = requests.get(url)

print(f"Status code: {response.status_code}")  # 200 = success
print(f"Content length: {len(response.text)} characters")
print(f"\nFirst 500 characters of HTML:\n{response.text[:500]}")

### 1.2 Parsing HTML with BeautifulSoup

Raw HTML is hard to work with as a string. **BeautifulSoup** parses it into a tree structure you can navigate and search.

**Key methods:**
- `soup.find('tag')` — find the first matching element
- `soup.find_all('tag')` — find all matching elements
- `element.text` — get the text content
- `element['attribute']` — get an attribute value (like `href` or `class`)

In [ ]:
soup = BeautifulSoup(response.text, 'html.parser')

# Find the page title
print(f"Page title: {soup.title.text}")

# Find all the book category links in the sidebar
sidebar = soup.find('ul', class_='nav-list')
categories = sidebar.find_all('a')

print(f"\nNumber of category links found: {len(categories)}")
print("\nFirst 10 categories:")
for cat in categories[:10]:
    print(f"  - {cat.text.strip()}")

---

## Part 2: Scraping Book Data

Now let's extract structured data from the site. Each book listing on the page contains:
- **Title**
- **Price**
- **Rating** (1-5 stars)
- **Availability**

### 2.1 Scraping a Single Page

**Exercise:** Extract book information from the first page of results.

In [ ]:
def parse_books_from_page(soup):
    """Extract book data from a BeautifulSoup page object."""
    books = []
    
    # Each book is in an <article> tag with class 'product_pod'
    articles = soup.find_all('article', class_='product_pod')
    
    # Map star rating words to numbers
    rating_map = {'One': 1, 'Two': 2, 'Three': 3, 'Four': 4, 'Five': 5}
    
    for article in articles:
        # Title is in the <a> tag inside <h3>
        title = article.h3.a['title']
        
        # Price is in a <p> tag with class 'price_color'
        price_text = article.find('p', class_='price_color').text
        price = float(price_text.replace('£', '').strip())
        
        # Rating is encoded as a CSS class on a <p> tag: e.g., 'star-rating Three'
        rating_class = article.find('p', class_='star-rating')['class'][1]
        rating = rating_map.get(rating_class, 0)
        
        # Availability
        avail_text = article.find('p', class_='instock').text.strip()
        in_stock = 'In stock' in avail_text
        
        books.append({
            'title': title,
            'price': price,
            'rating': rating,
            'in_stock': in_stock
        })
    
    return books

# Parse the first page
page1_books = parse_books_from_page(soup)
print(f"Books found on page 1: {len(page1_books)}")
print(f"\nFirst 3 books:")
for b in page1_books[:3]:
    print(f"  {b['title'][:50]:50s} | £{b['price']:5.2f} | {'★' * b['rating']}")

### 2.2 Scraping Multiple Pages

The site has 50 pages of books. Let's scrape all of them. This is a common pattern: identify the URL structure, then loop through pages.

**Important:** When scraping, always be respectful — add a short delay between requests to avoid overloading the server.

In [ ]:
all_books = []
base_url = "http://books.toscrape.com/catalogue/page-{}.html"

for page_num in range(1, 51):
    url = base_url.format(page_num)
    response = requests.get(url)
    
    if response.status_code != 200:
        print(f"Failed on page {page_num}: status {response.status_code}")
        break
    
    page_soup = BeautifulSoup(response.text, 'html.parser')
    page_books = parse_books_from_page(page_soup)
    all_books.extend(page_books)
    
    # Print progress every 10 pages
    if page_num % 10 == 0:
        print(f"  Scraped page {page_num}/50 ({len(all_books)} books so far)")
    
    # Be polite — wait briefly between requests
    time.sleep(0.3)

print(f"\nTotal books scraped: {len(all_books)}")

### 2.3 Scraping Book Categories

Each book belongs to a genre/category. To get this, we need to visit each book's detail page. Since that's 1000 requests, let's instead scrape the category pages to map books to genres.

**Exercise:** Scrape the list of categories and the number of books in each.

In [ ]:
# Fetch the main page to get category links
main_soup = BeautifulSoup(requests.get("http://books.toscrape.com/").text, 'html.parser')
sidebar = main_soup.find('ul', class_='nav-list')

# The first <li> is "Books" (all), nested <ul> contains genre categories
genre_links = sidebar.find('ul').find_all('a')

categories = []
for link in genre_links:
    name = link.text.strip()
    href = "http://books.toscrape.com/" + link['href']
    
    # Visit category page to get book count
    cat_response = requests.get(href)
    cat_soup = BeautifulSoup(cat_response.text, 'html.parser')
    results_text = cat_soup.find('form', class_='form-horizontal').find('strong').text
    count = int(results_text)
    
    categories.append({'category': name, 'book_count': count, 'url': href})
    time.sleep(0.2)

categories_df = pd.DataFrame(categories)
print(f"Total categories: {len(categories_df)}")
categories_df.sort_values('book_count', ascending=False).head(10)

---

## Part 3: Building a Clean Dataset

Now let's convert our scraped data into a proper Pandas DataFrame and clean it up.

### 3.1 Creating the DataFrame

In [ ]:
books_df = pd.DataFrame(all_books)

print(f"Shape: {books_df.shape}")
print(f"\nColumn types:\n{books_df.dtypes}")
print(f"\nMissing values:\n{books_df.isnull().sum()}")
books_df.head()

### 3.2 Data Cleaning

**Exercise:** 
1. Check for and handle any duplicate titles.
2. Create a `price_bucket` column that categorizes books as 'Budget' (< £20), 'Mid-range' (£20-40), or 'Premium' (> £40).
3. Create a `title_length` column with the number of characters in each title.

In [ ]:
# 1. Check duplicates
print(f"Duplicate titles: {books_df['title'].duplicated().sum()}")

# 2. Price buckets
def price_bucket(price):
    if price < 20:
        return 'Budget'
    elif price <= 40:
        return 'Mid-range'
    else:
        return 'Premium'

books_df['price_bucket'] = books_df['price'].apply(price_bucket)

# 3. Title length
books_df['title_length'] = books_df['title'].str.len()

print(f"\nPrice bucket distribution:")
print(books_df['price_bucket'].value_counts())

print(f"\nAverage title length: {books_df['title_length'].mean():.1f} characters")
books_df.head()

---

## Part 4: Exploratory Data Analysis

With a clean dataset in hand, let's explore it to uncover patterns and insights.

### 4.1 Price Distribution

**Exercise:** Visualize the distribution of book prices.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(books_df['price'], bins=25, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].set_xlabel('Price (£)')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of Book Prices')
axes[0].axvline(books_df['price'].mean(), color='red', linestyle='--', label=f"Mean: £{books_df['price'].mean():.2f}")
axes[0].axvline(books_df['price'].median(), color='orange', linestyle='--', label=f"Median: £{books_df['price'].median():.2f}")
axes[0].legend()

# Box plot
axes[1].boxplot(books_df['price'], vert=True)
axes[1].set_ylabel('Price (£)')
axes[1].set_title('Price Box Plot')

plt.tight_layout()
plt.show()

# Summary stats
print(books_df['price'].describe())

### 4.2 Rating Analysis

**Exercise:** 
1. How are ratings distributed?
2. Is there a relationship between price and rating?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Rating distribution
rating_counts = books_df['rating'].value_counts().sort_index()
axes[0].bar(rating_counts.index, rating_counts.values, color='goldenrod', edgecolor='white')
axes[0].set_xlabel('Rating (stars)')
axes[0].set_ylabel('Number of Books')
axes[0].set_title('Distribution of Book Ratings')
axes[0].set_xticks([1, 2, 3, 4, 5])

# 2. Price vs. Rating
axes[1].boxplot([books_df[books_df['rating'] == r]['price'] for r in range(1, 6)],
                labels=[1, 2, 3, 4, 5])
axes[1].set_xlabel('Rating (stars)')
axes[1].set_ylabel('Price (£)')
axes[1].set_title('Price Distribution by Rating')

plt.tight_layout()
plt.show()

# Correlation
corr = books_df['price'].corr(books_df['rating'])
print(f"Correlation between price and rating: {corr:.3f}")

### 4.3 Price Bucket Breakdown

**Exercise:** Compare the average rating across price buckets. Do more expensive books tend to be rated higher?

In [ ]:
bucket_stats = books_df.groupby('price_bucket').agg(
    count=('title', 'count'),
    avg_price=('price', 'mean'),
    avg_rating=('rating', 'mean'),
    avg_title_length=('title_length', 'mean')
).round(2)

# Reorder
bucket_stats = bucket_stats.reindex(['Budget', 'Mid-range', 'Premium'])
print(bucket_stats)

### 4.4 Category Analysis

Let's visualize the distribution of books across genres using the category data we scraped earlier.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

# Sort and plot horizontal bar chart
cat_sorted = categories_df.sort_values('book_count', ascending=True)
ax.barh(cat_sorted['category'], cat_sorted['book_count'], color='steelblue', alpha=0.8)
ax.set_xlabel('Number of Books')
ax.set_title('Books by Category')
plt.tight_layout()
plt.show()

### 4.5 Title Length Exploration

**Exercise:** Does title length vary by rating? Some research suggests that shorter, punchier titles might be associated with different reception patterns.

Create a visualization and compute summary stats to investigate.

In [ ]:
fig, ax = plt.subplots()

# Violin plot of title length by rating
data_by_rating = [books_df[books_df['rating'] == r]['title_length'].values for r in range(1, 6)]
parts = ax.violinplot(data_by_rating, positions=range(1, 6), showmeans=True, showmedians=True)
ax.set_xlabel('Rating (stars)')
ax.set_ylabel('Title Length (characters)')
ax.set_title('Title Length Distribution by Rating')
ax.set_xticks(range(1, 6))
plt.tight_layout()
plt.show()

# Stats
print("Mean title length by rating:")
print(books_df.groupby('rating')['title_length'].mean().round(1))

---

## Part 5: Putting It All Together

**Exercise:** Create a summary report of the dataset. Produce a single figure with 4 subplots that gives a complete overview of the books data:
1. Price distribution (histogram)
2. Rating distribution (bar chart)
3. Price vs. rating (scatter plot)
4. Top 10 categories by book count (horizontal bar chart)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Price distribution
axes[0, 0].hist(books_df['price'], bins=25, color='steelblue', edgecolor='white', alpha=0.8)
axes[0, 0].set_xlabel('Price (£)')
axes[0, 0].set_ylabel('Count')
axes[0, 0].set_title('Price Distribution')

# 2. Rating distribution
rc = books_df['rating'].value_counts().sort_index()
axes[0, 1].bar(rc.index, rc.values, color='goldenrod', edgecolor='white')
axes[0, 1].set_xlabel('Rating (stars)')
axes[0, 1].set_ylabel('Count')
axes[0, 1].set_title('Rating Distribution')
axes[0, 1].set_xticks([1, 2, 3, 4, 5])

# 3. Price vs. Rating (jittered scatter)
jitter = np.random.normal(0, 0.15, size=len(books_df))
axes[1, 0].scatter(books_df['rating'] + jitter, books_df['price'], alpha=0.3, s=15, color='teal')
axes[1, 0].set_xlabel('Rating (stars)')
axes[1, 0].set_ylabel('Price (£)')
axes[1, 0].set_title('Price vs. Rating')

# 4. Top 10 categories
top10 = categories_df.nlargest(10, 'book_count').sort_values('book_count', ascending=True)
axes[1, 1].barh(top10['category'], top10['book_count'], color='coral', alpha=0.8)
axes[1, 1].set_xlabel('Number of Books')
axes[1, 1].set_title('Top 10 Categories')

plt.suptitle('Books Dataset Overview', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

---

## Wrap-Up

In this notebook you practiced:

- **Web scraping** — making HTTP requests, parsing HTML with BeautifulSoup, and extracting structured data
- **Data cleaning** — handling duplicates, creating derived features, and categorizing values
- **Exploratory data analysis** — distributions, correlations, group comparisons, and multi-panel visualizations

Key takeaways:
- Always be respectful when scraping — add delays and check the site's terms of use
- Clean data before analyzing it — check for missing values, duplicates, and data types
- EDA should tell a story — let the data guide your questions

The next notebook will introduce **regression models** for making predictions from data.